In [9]:
import json
import os
import subprocess
import sys
import threading
from pathlib import Path

import py3Dmol
from sample.sample_config import (
    GenerationParams,
    SampleCheckpointParams,
    SampleConfig,
    SampleOutputParams,
)
from train.train_config import (
    CheckpointParams,
    TrainLoaderConfig,
    TrainConfig,
    TrainingParams,
)

PALLATOM_ROOT = Path.cwd().resolve()
if str(PALLATOM_ROOT) not in sys.path:
    sys.path.insert(0, str(PALLATOM_ROOT))

PALLATOM_ROOT


PosixPath('/workspaces/diffusion/pallatom')

In [10]:
import torch

torch.cuda.is_available()

True

# PallAtom: Training & Analysis

1. **Configure** — tune hyperparameters and serialise `TrainConfig` to JSON
2. **Train** — launch `train_loop.py` as a subprocess
3. **Sample** — load the saved checkpoint and run EDM backbone sampling
4. **Visualise** — render sampled structures with py3Dmol

## 1 · Training Configuration

In [11]:
def run_subprocess(cmd, out: dict):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PALLATOM_ROOT)
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(PALLATOM_ROOT),
        env=env,
    )
    out["pid"] = proc.pid
    print(f"Subprocess PID: {proc.pid}")

    stdout, _ = proc.communicate()
    output_lines = stdout.strip().splitlines() if stdout else []

    if proc.returncode != 0:
        print([f"ERROR (exit {proc.returncode}):"] + output_lines)
        out["result"] = None
        return

    out["result"] = output_lines

In [12]:
tcfg = TrainConfig(
    training=TrainingParams(
        num_epochs=150,
        # pretrained_weights="pallatom_toy_best.pt",
        # resume_checkpoint="pallatom_toy_best.pt",
        ),
    checkpoint=CheckpointParams(checkpoint_path="pallatom_toy_best.pt"),
    train_loader=TrainLoaderConfig(max_seq_length=128)
    )
tcfg

TrainConfig(training=TrainingParams(num_epochs=150, lr=0.001, weight_decay=0.0001, grad_clip=10.0, pretrained_weights=None, resume_checkpoint=None, accumulated_token_budget=4096, lr_decay_steps=50000, lr_decay_factor=0.95, ema_decay=0.999), model=ModelParams(window_size=32, f_ref_dim=35, c_atom=128, c_pair=128, c_res=256, c_atompair=16, K_unit=8, max_residues=128, n_amino=20, n_blocks_atom_transformer_encoder=3, n_heads_atom_transformer_encoder=4, n_blocks_atom_transformer_decoder=3, n_heads_atom_transformer_decoder=4, n_pairformer_blocks_template_embedder=2, n_paiformer_heads_template_embedder=16), noise=NoiseScheduleParams(sigma_data=16.0, sigma_max=160, sigma_min=0.0004, P_mean=-1.2, P_std=1.5), distogram_template=TemplateDistogramParams(min_dist=3.25, max_dist=50.75, n_bins=39, overflow_bin=True, tok_emb_dim=32), distogram_residue=ResidueDistogramParams(min_dist=2.0, max_dist=22.0, n_bins=64, overflow_bin=True), distogram_atom=AtomDistogramParams(min_dist=0.0, max_dist=10.0, n_bins

In [13]:
config_json_path = PALLATOM_ROOT / "train" / "run_config.json"
_ = config_json_path.write_text(
    tcfg.model_dump_json(indent=2)
)

In [6]:
log_path = PALLATOM_ROOT / "train" / "train_logs.jsonl"
shard_dir: Path = PALLATOM_ROOT / "data" / "shards"

train_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "train" / "train_loop.py"),
            "--dataset_jsonl",        str(PALLATOM_ROOT / "data" / "chain_set.jsonl"),
            "--keys_for_splits_json",      str(PALLATOM_ROOT / "data" / "chain_set_splits.json"),
            "--config",      config_json_path,
            "--structlog_jsonl",    log_path,
            # "--debug_run",
            "--shard_dir", shard_dir,
        ]

training_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(train_cmd, training_out),
    daemon=True
).start()

Subprocess PID: 1786


In [ ]:
import torch

NUM_GPUS = torch.cuda.device_count()
torchrun = str(Path(sys.executable).parent / "torchrun")
ddp_log_path = PALLATOM_ROOT / "train" / "train_logs_ddp.jsonl"
shard_dir: Path = PALLATOM_ROOT / "data" / "shards"

ddp_train_cmd = [
    torchrun,
    f"--nproc_per_node={NUM_GPUS}",
    str(PALLATOM_ROOT / "train" / "train_loop.py"),
    "--dataset_jsonl",        str(PALLATOM_ROOT / "data" / "chain_set.jsonl"),
    "--keys_for_splits_json",      str(PALLATOM_ROOT / "data" / "chain_set_splits.json"),
    "--config",      config_json_path,
    "--structlog_jsonl",    ddp_log_path,
    "--debug_run",
    "--shard_dir", shard_dir,
    "--ddp",
]

ddp_training_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(ddp_train_cmd, ddp_training_out),
    daemon=True
).start()

Subprocess PID: 5485


['ERROR (exit 1):', 'W0708 21:08:59.681000 5485 torch/distributed/run.py:851] ', 'W0708 21:08:59.681000 5485 torch/distributed/run.py:851] *****************************************', 'W0708 21:08:59.681000 5485 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. ', 'W0708 21:08:59.681000 5485 torch/distributed/run.py:851] *****************************************', 'wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.', 'wandb: Currently logged in as: tlmattesonr to https://api.wandb.ai. Use `wandb login --relogin` to force relogin', 'wandb: setting up run n5oga949', 'wandb: Tracking run with wandb version 0.26.1', 'wandb: Run data is saved locally in /workspaces/diffusion/pallatom/wandb/run-20260708_210938-n5oga949', 'wandb: Run `wandb offline` to turn off syncing.', '

In [14]:
ckpt_path       = str(".." / tcfg.checkpoint.checkpoint_path)
ckpt_path

'../pallatom_toy_best.pt'

In [15]:
from sample.sample_config import SamplerParams


sample_output_path     = str(Path(config_json_path).with_name("samples.json"))
sample_cfg_path = str(Path(config_json_path).with_name("sample_config.json"))

scfg = SampleConfig(
    model=tcfg.model,
    noise=tcfg.noise,
    generation=GenerationParams(
        n_res=tcfg.test_loader.max_seq_length,
        n_samples=tcfg.test_loader.batch_size, # 16 GB VRAM

    ),
    checkpoint=SampleCheckpointParams(checkpoint_path=ckpt_path),
    output=SampleOutputParams(output_path=sample_output_path),
)

sample_config_json_path = PALLATOM_ROOT / "train" / "sample_config.json"
with open(sample_config_json_path, "w") as _f:
    _f.write(json.dumps(scfg.model_dump(), indent=2))

In [16]:
sample_log_path = PALLATOM_ROOT / "train" / "sample_logs.jsonl"
sample_cmd = [
            sys.executable, "-u",
            str(PALLATOM_ROOT / "sample" / "sampling.py"),
            "--config", sample_cfg_path,
            "--log_file", sample_log_path,
        ]
# does this not pipe logs out into a structlog
sample_out = dict()
threading.Thread(
    target=run_subprocess,
    args=(sample_cmd, sample_out),
    daemon=True
).start()

Subprocess PID: 112880


In [17]:
sample_out

{'pid': 112880,
 'result': ["\x1b2026-07-10T19:26:31.796962Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bconfig loaded                 \x1b \x1bconfig\x1b=\x1bPosixPath('/workspaces/diffusion/pallatom/train/sample_config.json')\x1b \x1bn_res\x1b=\x1b128\x1b \x1bn_samples\x1b=\x1b8\x1b",
  '\x1b2026-07-10T19:26:33.613024Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bmodel loaded                  \x1b \x1bcheckpoint\x1b=\x1b../pallatom_toy_best.pt\x1b \x1bdevice\x1b=\x1bcuda\x1b',
  '\x1b2026-07-10T19:26:34.019767Z\x1b [\x1b\x1binfo     \x1b[0m] \x1bsampling                      \x1b \x1bddim_steps\x1b=\x1b200\x1b \x1bn_res\x1b=\x1b128\x1b \x1bn_samples\x1b=\x1b8\x1b',
  '/workspaces/diffusion/pallatom/helpers/alignment.py:344: UserWarning: torch.qr is deprecated in favor of torch.linalg.qr and will be removed in a future PyTorch release.',
  "The boolean parameter 'some' has been replaced with a string parameter 'mode'.",
  'Q, R = torch.qr(A, some)',
  'should be replaced with',
  "Q, R = torch.linalg.qr

In [19]:
# lets load the pdb files from samples.json

# Open the file in read mode ('r')
with open('train/samples.json') as file:
    # Use json.load() to read and parse the file
    data = json.load(file)

# Now 'data' is a standard Python object (dict or list)
print(len(data))
pdb_str = data[1]
view = py3Dmol.view(
    width=600, height=600, linked=True , viewergrid=(1, 1))
view.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})
style = {"cartoon": {'color': 'spectrum'}}

view.addModelsAsFrames(pdb_str, viewer=(0, 0))
view.setStyle({'model': -1}, style, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

view.render()


# and then show them in py3dmol

8


3Dmol.js failed to load for some reason. Please check your browser console for error messages.